# 강의 05 · 실습 3 — 운영 장치 · (6) 고난도 III


## 1. 문제상황

- 구름월드 운영팀은 안내문을 새 판으로 바꾸려 합니다. 새 판은 규정집 전문을 안내문에 넣어 답이 더 정확해졌지만, 호출마다 보내는 글이 훨씬 길어졌습니다.
- 개발팀은 「답이 좋아졌으니 배포하자」고 하고, 운영팀은 「비용이 얼마나 늘었는지 모르면 배포할 수 없다」고 합니다.
- 지금은 새 판의 비용을 잰 적이 없고, 캐시를 붙이면 비용이 얼마나 줄어드는지도 모릅니다.
- 운영팀은 같은 질문 5개로 이전 판·새 판·새 판(캐시 적용)의 총비용을 재고, 새 판(캐시 적용)의 총비용이 이전 판의 1.5배를 넘으면 배포를 자동으로 막는 장치를 원합니다.


## 2. 문제와 목표

- **문제**: 안내문 변경이 비용에 미치는 영향을 재지 않고 배포 결정을 내리려 하며, 캐시의 효과도 모릅니다.
- **목표**
  - 같은 질문 5개로 세 가지 구성의 호출 비용과 캐시 히트를 표로 출력합니다.
    - 구성 세 개: 이전 판은 서비스 코드의 짧은 안내문, 새 판 1회차는 FAQ를 열두 번 반복해 붙인 긴 안내문(맨 앞에 실행 시각 표시), 새 판 2회차는 같은 긴 안내문을 한 번 더(캐시 적용). 모든 호출은 기본 모델로 합니다
    - 질문 5개는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다
    - 메시지 목록은 시스템 메시지와 사용자 메시지 두 개로 직접 만듭니다
  - 구성별 총비용과 이전 판 대비 배율을 계산해, 새 판(캐시 적용)의 배율이 1.5를 넘으면 「차단」, 아니면 「통과」를 출력하는 비용 회귀 게이트를 만듭니다.
    - 배율: 새 판 2회차 총비용 ÷ 이전 판 총비용. 마지막 줄은 「새 판(캐시 적용) / 이전 판 = N배 → 통과 또는 차단」 형식
- **목표 달성 여부의 판정 기준**:
  - 표에 구성 3개 × 질문 5개 = 15줄이 출력되고,
  - 새 판 첫 실행의 총비용이 이전 판보다 크며,
  - 새 판 두 번째 실행의 캐시 히트가 0보다 크고 총비용이 첫 실행보다 작습니다.
  - 마지막 줄에 「새 판(캐시 적용) / 이전 판 = N배 → 통과 또는 차단」이 출력됩니다.


## 3. 워크플로우 다이어그램


## 4. 단계별 요구사항

이 단에서는 요구사항을 주지 않습니다. 「1. 문제상황」「2. 문제와 목표」와 「3. 워크플로우 다이어그램」에 직접 그린 다이어그램을 보고 요구사항을 번호 목록으로 적은 뒤 「6. 코드 — 스텝바이스텝」의 코드를 작성합니다.


## 5. 코드 골격

(이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다. 쓰지 않는 단은 이유를 적습니다.)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델 이름 세 개를 정합니다. 이 실습의 모델 호출은 `litellm.completion`을 직접 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- 모델 세 개는 기본 모델(`PRIMARY`), 경량 모델(`CHEAP`), 고성능 모델(`HIGH`)입니다. 모델 이름은 공급자 이름을 앞에 붙인 문자열 그대로 쓰고, 별칭이나 중계 서버는 쓰지 않습니다.
- `litellm.suppress_debug_info = True`와 `logging` 설정 한 줄은 오류가 났을 때 litellm이 화면에 출력하는 안내 배너와 오류 로그를 끕니다. 동작에는 영향이 없습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import logging
import os
import time   # time — time.strftime("%H%M%S")로 긴 안내문 맨 앞의 실행 시각 표시를 만듭니다
import warnings

from dotenv import load_dotenv, find_dotenv

import litellm
from langsmith import Client, traceable   # Client — Client().flush()로 남은 런을 LangSmith로 보냅니다
from langsmith.run_helpers import get_current_run_tree

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")   # 추적 라이브러리가 내는 직렬화 경고를 화면에서 감춥니다
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex03"

PRIMARY = "openai/gpt-5.6-luna"
CHEAP = "openai/gpt-4o-mini"
HIGH = "openai/gpt-5.6-terra"
print("모델 세 개:", PRIMARY, CHEAP, HIGH)

# 주어진 자료: 처리할 질문 목록 QUESTIONS — 값을 그대로 씁니다
QUESTIONS = ["자유이용권 환불이 되나요?", "운영 시간이 어떻게 되나요?", "야간개장 때 퍼레이드 하나요?",
             "주차 요금은 얼마인가요?", "안녕하세요!"]


운영 장치를 붙일 안내 서비스입니다. 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_messages`가 질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만들고, 모델 호출 함수 `litellm.completion`은 `traceable`로 감싸 llm 런에 기록되어 있습니다(답한 모델 이름을 런 메타데이터에 적습니다). 서비스가 도는 것을 먼저 확인합니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

GUIDE = ("너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
         "인사말에는 짧은 인사로 답한다. FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.")


def make_messages(question: str, guide: str = GUIDE) -> list:
    """질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만든다."""
    return [{"role": "system", "content": guide + "\n=== FAQ ===\n" + FAQ},
            {"role": "user", "content": question}]


_completion = litellm.completion


@traceable(run_type="llm", name="litellm.completion", metadata={"ls_provider": "openai"})
def completion(**kwargs):
    """이 실습에서 쓰는 관측 계측. 답한 모델 이름을 런 메타데이터에 적는다."""
    res = _completion(**kwargs)
    get_current_run_tree().metadata["ls_model_name"] = res.model
    return res


litellm.completion = completion

Q = "자유이용권 환불이 되나요?"
res = litellm.completion(model=PRIMARY, messages=make_messages(Q))
print(res.model, "→", res.choices[0].message.content[:60])

In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 표에 구성 3개 × 질문 5개 = 15줄이 출력됩니다.
2. 새 판 첫 실행의 총비용이 이전 판보다 큽니다. 안내문이 길어진 만큼 입력 비용이 늘었습니다.
3. 새 판 두 번째 실행의 캐시 히트가 0보다 크고, 총비용이 첫 실행보다 작습니다.
4. 마지막 줄에 배율과 「통과」 또는 「차단」이 출력됩니다.

네 가지가 모두 확인되면 완성입니다. 통과·차단이 어느 쪽이든, 배율 숫자가 배포 결정의 근거입니다.
